# 📖 Notebook 2: UUID, ULID, and KSUID Compared

Notebook 1 showed *why* we want coordination-free, time-ordered IDs.

Now let's look at four real formats people actually use and **implement each one from scratch** so nothing feels like magic.

## Learning Objectives

- Know the bit layout of UUIDv4, UUIDv7, ULID, and KSUID
- Generate each one with only Python's standard library
- Understand the trade-offs: size, sortability, entropy, and ecosystem support
- Pick the right ID type for a given problem


## 🛠️ Setup

This lab uses **only the Python standard library + `pydantic`**. No Docker, no Redis, no Postgres.

```bash
cd 01-foundations/id-generation
uv sync
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


## 🔢 Refresher: bits, bytes, and encodings

All four formats are just **bit strings** — what differs is:

1. **How many bits** (128 for UUID/ULID, 160 for KSUID, 64 for Snowflake)
2. **What's in the bits** (random? timestamp? counter?)
3. **How the bits are displayed** (hex, Base32, Base62)

We'll handle each layout with plain `int`s and `bytes`, and only reach for encoding helpers when we display them.


## 1️⃣ UUIDv4 — 128 bits of randomness

**Layout:** 122 random bits + 6 fixed bits (version `0100` and variant `10`).

**Display:** 32 hex chars with dashes → 36 chars total, e.g. `550e8400-e29b-41d4-a716-446655440000`.

Let's use the built-in `uuid` module, then roll our own to show there's no magic.


In [ ]:
import uuid, os

# Built-in:
print("stdlib uuid4:", uuid.uuid4())

def my_uuid4() -> str:
    """RFC 4122 v4: 16 random bytes with version and variant bits patched in."""
    b = bytearray(os.urandom(16))
    b[6] = (b[6] & 0x0F) | 0x40   # set version = 4  (top nibble of byte 6)
    b[8] = (b[8] & 0x3F) | 0x80   # set variant = 10xxxxxx (RFC 4122)
    hx = b.hex()
    return f"{hx[0:8]}-{hx[8:12]}-{hx[12:16]}-{hx[16:20]}-{hx[20:32]}"

for _ in range(3):
    print("my_uuid4:  ", my_uuid4())


### Collision probability (birthday paradox)

With 122 random bits, you'd need to generate about **2⁶¹ ≈ 2.3 × 10¹⁸** UUIDs before a 50% collision chance. That's more IDs than stars in the observable universe. Safe.


## 2️⃣ UUIDv7 — time-ordered UUID (RFC 9562)

UUIDv7 is the **newer, sortable** UUID variant, standardised in 2024.

**Layout (128 bits total):**

```
|  48 bits  | 4 | 12 bits | 2 |       62 bits        |
|  unix_ms  | v |  rand_a | r |        rand_b         |
```

- **48 bits** = Unix timestamp in **milliseconds** (good until year ~10889)
- **4 bits** = version `0111` (7)
- **12 bits** = random
- **2 bits** = variant `10`
- **62 bits** = random

Total random bits = 12 + 62 = 74. Still astronomical collision resistance, and now **sortable by creation time**.


In [ ]:
import os, time

def my_uuid7() -> str:
    # 1) 48-bit millisecond timestamp
    unix_ms = int(time.time() * 1000) & ((1 << 48) - 1)
    # 2) Pull 10 random bytes (80 random bits, more than the 74 we need)
    rand = int.from_bytes(os.urandom(10), 'big')
    # 3) Compose the 128-bit integer
    #    bits 127..80 = unix_ms  (48 bits)
    #    bits 79..76  = version = 7
    #    bits 75..64  = rand_a  (12 bits)
    #    bits 63..62  = variant = 0b10
    #    bits 61..0   = rand_b  (62 bits)
    rand_a = (rand >> (80 - 12)) & 0xFFF            # top 12 random bits
    rand_b = rand & ((1 << 62) - 1)                 # bottom 62 random bits

    val  = unix_ms << 80
    val |= (0x7 << 76)
    val |= (rand_a << 64)
    val |= (0b10 << 62)
    val |= rand_b

    # 4) Format as the standard UUID string
    hx = f"{val:032x}"
    return f"{hx[0:8]}-{hx[8:12]}-{hx[12:16]}-{hx[16:20]}-{hx[20:32]}"

ids = []
import time as _t
for _ in range(5):
    ids.append(my_uuid7())
    _t.sleep(0.002)
for i in ids:
    print(i)

print()
print("Sorted == creation order?", ids == sorted(ids))


Notice the **first 12 hex chars are always increasing** — that's the 48-bit millisecond timestamp. That's what makes UUIDv7 sortable.


## 3️⃣ ULID — Universally Unique Lexicographically Sortable Identifier

ULID uses the same 128 bits as UUID but splits them differently:

```
|   48 bits   |       80 bits        |
|  unix_ms    |     randomness       |
```

**Display:** 26 chars of **Crockford Base32** (0-9 A-Z except I, L, O, U), case-insensitive, URL-safe.

Example: `01ARZ3NDEKTSV4RRFFQ69G5FAV`

Let's implement it.


In [ ]:
import os, time

# Crockford's Base32 alphabet (no I, L, O, U to avoid confusion)
CROCKFORD = "0123456789ABCDEFGHJKMNPQRSTVWXYZ"

def _b32_encode(n: int, length: int) -> str:
    """Encode a non-negative int as Crockford Base32 of exactly `length` chars."""
    out = []
    for _ in range(length):
        out.append(CROCKFORD[n & 0x1F])
        n >>= 5
    return ''.join(reversed(out))

def my_ulid() -> str:
    unix_ms = int(time.time() * 1000) & ((1 << 48) - 1)
    randomness = int.from_bytes(os.urandom(10), 'big')  # 80 bits
    # 48-bit time -> 10 Base32 chars; 80-bit rand -> 16 Base32 chars; total = 26
    return _b32_encode(unix_ms, 10) + _b32_encode(randomness, 16)

ids = []
for _ in range(5):
    ids.append(my_ulid())
    time.sleep(0.002)
for i in ids:
    print(i)

print()
print("Sorted == creation order?", ids == sorted(ids))
print("Length of each:", len(ids[0]), "chars")


### Monotonicity within the same millisecond

If two ULIDs are generated in the **same** millisecond on the same machine, only the 80 random bits differ — and nothing guarantees the second one is larger.

Real ULID libraries fix this by, within the same ms, taking the previous random part and **incrementing it** instead of randomising again. A simple implementation:


In [ ]:
class MonotonicUlid:
    def __init__(self):
        self._last_ms = 0
        self._last_rand = 0

    def new(self) -> str:
        ms = int(time.time() * 1000) & ((1 << 48) - 1)
        if ms == self._last_ms:
            # Same millisecond: increment the random part to preserve order.
            self._last_rand = (self._last_rand + 1) & ((1 << 80) - 1)
        else:
            self._last_ms = ms
            self._last_rand = int.from_bytes(os.urandom(10), 'big')
        return _b32_encode(ms, 10) + _b32_encode(self._last_rand, 16)

gen = MonotonicUlid()
burst = [gen.new() for _ in range(5)]
for b in burst:
    print(b)
print("Monotonically increasing?", burst == sorted(burst))


## 4️⃣ KSUID — K-Sortable Unique Identifier

KSUID was created by Segment. **160 bits** total:

```
|  32 bits   |       128 bits        |
|  unix_sec  |     randomness        |
```

Key differences vs ULID:

- Timestamp is in **seconds** (not ms), with a **custom epoch** (14 May 2014)
- **128 bits** of randomness (vs 80 in ULID) → more collision resistance per second
- Displayed as **27 chars of Base62** (0-9 A-Z a-z)
- Total size: **20 bytes** (the largest of the group)


In [ ]:
import os, time

KSUID_EPOCH = 1400000000  # May 13, 2014 (Segment's chosen epoch)

BASE62 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

def _base62_encode(n: int, length: int) -> str:
    out = []
    for _ in range(length):
        out.append(BASE62[n % 62])
        n //= 62
    return ''.join(reversed(out))

def my_ksuid() -> str:
    ts = int(time.time()) - KSUID_EPOCH  # 32-bit field
    payload = int.from_bytes(os.urandom(16), 'big')  # 128 bits
    value = (ts << 128) | payload
    # 160-bit value -> 27 Base62 chars
    return _base62_encode(value, 27)

ids = [my_ksuid() for _ in range(5)]
for i in ids:
    print(i)

print()
print("Length:", len(ids[0]), "chars")


## 🧪 Side-by-side comparison

Let's validate the claim that UUIDv7, ULID, and KSUID all sort by time — and UUIDv4 does not.


In [ ]:
import time

def burst(generator, n=8, delay=0.002):
    out = []
    for _ in range(n):
        out.append(generator())
        time.sleep(delay)
    return out

def check(name, ids):
    is_sorted = ids == sorted(ids)
    print(f"{name:10s} sorted==creation? {is_sorted}   len={len(ids[0])}")

check("uuid4",  [str(uuid.uuid4()) for _ in range(8)])
check("uuid7",  burst(my_uuid7))
check("ulid",   burst(my_ulid))
check("ksuid",  burst(my_ksuid))


### 🤔 Wait — why did KSUID fail the sort test?

Look carefully: our `burst()` helper sleeps only 2 ms between generations. But KSUID's timestamp is in **seconds**, not milliseconds!

So inside the same second, KSUIDs differ **only in their 128 random bits** — and random bits don't sort.

If we had spaced the bursts ≥1 second apart, KSUIDs would sort. This is the real trade-off: KSUID is "**k**-sortable" (sortable at *roughly* the granularity of seconds), not strictly sortable like Snowflake or monotonic ULID.

ULID (ms precision) and UUIDv7 (ms precision + monotonic options) give finer ordering.


## 📋 Cheat sheet

| Format   | Bits | Text len | Encoding     | Timestamp | Random bits | Sortable | Needs lib |
|----------|------|----------|--------------|-----------|-------------|----------|-----------|
| UUIDv4   | 128  | 36       | hex+dashes   | —         | 122         | ❌       | stdlib    |
| UUIDv7   | 128  | 36       | hex+dashes   | 48-bit ms | 74          | ✅ (ms)   | stdlib (3.14+) or easy to roll |
| ULID     | 128  | 26       | Base32       | 48-bit ms | 80          | ✅ (ms)   | 3rd party |
| KSUID    | 160  | 27       | Base62       | 32-bit s  | 128         | ⚠️ (s)    | 3rd party |
| NanoID   | ~126 | 21       | Base64-url   | —         | ~126        | ❌       | tiny lib (or 6 lines, see Notebook 4) |

### Rule of thumb

- **UUIDv4** — default. Anywhere you don't care about sort order (auth tokens, internal keys).
- **UUIDv7** — the "new default" for primary keys in modern databases. Standardised, sortable, drop-in for existing UUID columns.
- **ULID** — when you want a short, human-friendly, case-insensitive string (log IDs, URLs).
- **KSUID** — when you want even more entropy and a 100+ year timeline (high-volume event streams).
- **NanoID** — when you want a *short* URL-friendly random ID (covered in Notebook 4).

In the next notebook we'll look at the smallest of the family: **64-bit Snowflake IDs**, then in Notebook 4 we'll cover the production patterns you actually need to ship these.


## 📦 Bonus: a `pydantic` model that accepts any of them

Since the repo leans on `pydantic` for data validation, here's a quick model that treats any of these IDs as a validated string.


In [ ]:
from pydantic import BaseModel, field_validator
import re

UUID_RE  = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$")
ULID_RE  = re.compile(r"^[0-9A-HJKMNP-TV-Z]{26}$")
KSUID_RE = re.compile(r"^[0-9A-Za-z]{27}$")

class Event(BaseModel):
    id: str
    payload: str

    @field_validator("id")
    @classmethod
    def looks_like_an_id(cls, v: str) -> str:
        if UUID_RE.match(v) or ULID_RE.match(v) or KSUID_RE.match(v):
            return v
        raise ValueError(f"{v!r} does not look like a UUID/ULID/KSUID")

print(Event(id=my_uuid7(), payload="hello"))
print(Event(id=my_ulid(),  payload="world"))
print(Event(id=my_ksuid(), payload="again"))

try:
    Event(id="not-an-id", payload="oops")
except Exception as e:
    print("Rejected:", e)
